# 1.Package imports section

In [0]:
import re
import logging
import pyspark.sql.functions as F
from pyspark.sql import Window

# 2.Dataset configurations

In [0]:
# logging configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Silver_Layer_CPI")

#silver tables config
ds_config = {
        "silver_table": "cpt_utility_catalog.silver.silver_cpi_cleaned",
        "bronze_table": "cpt_utility_catalog.bronze.bronze_cpi_raw",
        "changes": {
            "headers": {
                "column_mapping": {
                    "H01": "survey_code",
                    "H03": "series_identifier",
                    "H04": "category",
                    "H05": "subcategory",
                    "H06": "decile",
                    "H13": "geographic_area",
                    "H17": "unit_of_measure",
                    "H18": "base_period",
                    "H24": "start_period",
                    "H25": "frequency",
                },
                 "date_header_mapping":{
                    "pattern": "^MO\\d{6}",
                    "after_pattern": "\\d{4}-\\d{2}",
                    "format": "yyyy-MM",
                },
            },
            "columns": {
                 "data_types":{
                    "survey_code": "string",
                    "series_identifier": "string",
                    "category": "string",
                    "subcategory": "string",
                    "decile": "int",
                    "geographic_area": "string",
                    "unit_of_measure": "string",
                    "base_period": "string",
                    "start_period": "date",
                    "frequency": "string",
                    "date": "date",
                    "index_value": "decimal(5,1)"
                },
                "columns_to_drop": ["H02"],
                "fill_na_value": None, 
            },
            "trim": True,
            "drop_columns": True,
            "drop_duplicates": True,
            "write_to_table": True,
            "rename_headers": True,
            "unpivot": True,
            "add_id": True,
            "cast_type": True,
            "col_cleanse": True,
        },
    }

df_new = spark.read.table(ds_config["bronze_table"])
changes = ds_config["changes"]
hdr_config = ds_config["changes"]["headers"]
col_config = ds_config["changes"]["columns"]

logger.info("Silver layer CPI table configuration loaded")

# 3. Dataset Cleaning

## 3.1 Drop Columns

In [0]:
if changes["drop_columns"]:
    logger.info("Droping column(s)")
    # obtain list of columns to delete
    drop_columns = col_config.get("columns_to_drop", [])
    # iterate through list of columns to delete and delete
    if isinstance(drop_columns, list):
        df_new = df_new.drop(*drop_columns)

    logger.info(f"\t- Column(s) successfully dropped")

##3.2 Rename Headers

In [0]:
if changes["rename_headers"]:
    logger.info("Renaming Header(s)")
    # obtain dictionary of columns to rename and rename them 
    column_mapping = hdr_config.get("column_mapping", {})
    if isinstance(column_mapping, dict):
        for old_col, new_col in column_mapping.items():
            df_new = df_new.withColumnRenamed(old_col, new_col)
    # obtain dated header dictionary and rename to yyyy-MM format
    date_header_mapping = hdr_config.get("date_header_mapping", {})
    if isinstance(date_header_mapping, dict):
        pattern = date_header_mapping.get("pattern", "")
        
        for col in df_new.columns:
            if re.match(pattern, col):
                new_name = f"{col[4:]}-{col[2:4]}"
                df_new = df_new.withColumnRenamed(col, new_name)
    logger.info("\t- Header(s) renamed")

## 3.3 Upivoting Table

In [0]:
if changes["unpivot"]:
    logger.info("Unpivoting CPI wide time-series dataset")

    # 1. Separate your metadata descriptor columns from your monthly date columns
    # All metadata columns are non-date strings; date columns match 'YYYY-MM' pattern
    metadata_columns = [
        "survey_code",
        "series_identifier",
        "category",
        "subcategory",
        "decile",
        "geographic_area",
        "unit_of_measure",
        "base_period",
        "start_period",
        "frequency",
    ]

    # Dynamically grab all columns that look like dates (YYYY-MM)
    monthly_columns = [col for col in df_new.columns if col not in metadata_columns]

    # transforming from wide to a long tale
    df_new = df_new.unpivot(
        ids=metadata_columns,
        values=monthly_columns,
        variableColumnName="date", 
        valueColumnName="index_value", 
    )

    logger.info("\t - CPI dataset successfully unpivoted and formatted")

## 3.3 Trim Whitespace

In [0]:
if changes["trim"]:    
    logger.info("Trimming whitespace(s)")
    # trim whitespace on all row cells sicne it's all string, before we cast type
    df_new = df_new.select([F.trim(F.col(col)).alias(col) if data_type == "string" else F.col(col) for col, data_type in df_new.dtypes])
    logger.info("\t- Whitespace(s) trimmed")

## 3.4 Column Cleansing

In [0]:
if changes["col_cleanse"]:
    logger.info("Cleaning Malformed Column row(s)")
    # turning start_period into convertable format of date data type
    df_new = df_new.withColumn("start_period", F.translate(F.col("start_period"), " ", "-"))
    logger.info("\t- Malformed column row(s) cleaned")

## 3.5 Cast Data Types

In [0]:
if changes["cast_type"]:

    logger.info("Casting data type(s)")
    dt_transformations = {}
    dt_types_map = col_config.get("data_types",{})
    date_header_map = hdr_config.get("date_header_mapping", {})

    # cast data types of fixed column names
    if isinstance(dt_types_map, dict):
        for col, data_type in dt_types_map.items():
            dt_transformations[col] = F.col(col).cast(data_type)

    df_new = df_new.withColumns(dt_transformations)
    logger.info("\t- Data type(s) casted")

## 3.6 Adding Primary Key

In [0]:
if changes["add_id"]:
    # creating primary key column with md5 hash of main columns   
    logger.info("Adding ID(s)")   
    df_new = df_new.withColumn(
        "id",
        F.md5(
            F.concat_ws(
                "|",
                F.col("series_identifier"),
                F.col("geographic_area"),
                F.col("category"),
                F.col("date"),
            )
        ),
    )
    logger.info("\t- ID(s) added")

## 3.7 Drop Duplicates

In [0]:
if changes["drop_duplicates"]:

    logger.info("Dropping Duplicate(s)")
    # Check all columns except your natural keys for nulls
    exclude_cols = ["series_identifier", "geographic_area", "date"]
    cols_to_check = [c for c in df_new.columns if c not in exclude_cols]

    # Count how many NULLs exist in each row across the metric/descriptor columns
    null_count_expr = sum(
        [F.when(F.col(c).isNull(), 1).otherwise(0) for c in cols_to_check]
    )

    df_new = df_new.withColumn("null_count", null_count_expr)

    # Create a window grouped by the TRUE grain, ordered by null_count ASCENDING
    window_spec = Window.partitionBy(
        "series_identifier", "geographic_area", "date"
    ).orderBy(F.col("null_count").asc())

    # Filter to keep only the best row per unique series/region/date and clean up
    df_deduped = (
        df_new.withColumn("row_num", F.row_number().over(window_spec))
        .filter(F.col("row_num") == 1)
        .drop("row_num", "null_count")
    )

    df_new = df_deduped
    logger.info("\t - Duplicate(s) Dropped")

# 4. Writing To Silver Layer

In [0]:
if changes["write_to_table"]:
    
    table_name = ds_config["silver_table"]

    logger.info("Starting Databricks Delta Lake Upsert Process via Spark SQL")

    # Register the incoming DataFrame as a temporary SQL view
    df_new.createOrReplaceTempView("temp_cpi_new")

    # Check if the target table already exists in Databricks
    if spark.catalog.tableExists(table_name):
        # Perform the SQL MERGE (Upsert)
        spark.sql(f"""
                MERGE INTO {table_name} AS target
                USING temp_cpi_new AS source
                ON target.id = source.id
                WHEN MATCHED THEN 
                    UPDATE SET *
                WHEN NOT MATCHED THEN 
                    INSERT *
            """)

        logger.info(f"\t - Successfully merged updates into table: {table_name}")

    else:
        # Initialize the table on the first run using Spark SQL
        spark.sql(f"""
                CREATE TABLE {table_name} 
                USING DELTA 
                AS SELECT * FROM temp_cpi_new
            """)

    logger.info(f"\t - Table {table_name} created and initialized successfully.")